# Lab 4 — Tool-Call Regression Testing

**Required challenge · 45 minutes**

Turn function-tool selection and arguments into a release gate. Exact local assertions protect hard contracts; Foundry Tool Call Accuracy adds semantic judgment about whether the calls satisfy the request.

**Artifact:** a passing local regression suite and a Foundry comparison run.

## Why two layers?

A deterministic assertion can prove that `qualification='LV-A'`. An LLM-assisted evaluator can judge whether calling `find_available_crew` was sensible for the user's request. Use both; never replace authorization or safety checks with an LLM score.

This lab evaluates synthetic **function tools**. Current Foundry agent evaluators have limited support for trajectories containing Azure AI Search and several other built-in tools.

In [ ]:
import os
import re
import time
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

def load_repo_env():
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        candidate = folder / '.env'
        if candidate.exists():
            load_dotenv(candidate)
            return candidate
    return None

load_repo_env()
endpoint = os.getenv('FOUNDRY_PROJECT_ENDPOINT') or os.getenv('AZURE_AI_PROJECT_ENDPOINT')
model_deployment = os.getenv('FOUNDRY_MODEL') or os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')
team_id = os.getenv('WORKSHOP_TEAM_ID', '').strip()
participant_id = os.getenv('WORKSHOP_PARTICIPANT_ID', '').strip()
configured_namespace = os.getenv('WORKSHOP_RESOURCE_NAMESPACE', '').strip()
raw_namespace = configured_namespace or team_id or participant_id
resource_namespace = re.sub(r'[^a-z0-9-]+', '-', raw_namespace.lower()).strip('-')[:32]
if not endpoint or not model_deployment or not resource_namespace:
    raise ValueError('Missing Foundry endpoint, model deployment, or namespace')
if tuple(int(p) for p in version('azure-ai-projects').split('.')[:2]) < (2, 2):
    raise RuntimeError('This lab targets azure-ai-projects>=2.2.0')
print({'namespace': resource_namespace, 'team': team_id, 'participant': participant_id})

## 1. Function-tool definitions

In [ ]:
tool_definitions = [
    {
        'name': 'lookup_incident',
        'description': 'Look up a synthetic incident by its exact identifier.',
        'parameters': {
            'type': 'object',
            'properties': {'incident_id': {'type': 'string', 'pattern': '^SIM-[0-9]{4}$'}},
            'required': ['incident_id'],
            'additionalProperties': False,
        },
    },
    {
        'name': 'lookup_procedure',
        'description': 'Look up a synthetic work procedure by exact procedure identifier.',
        'parameters': {
            'type': 'object',
            'properties': {'procedure_id': {'type': 'string', 'pattern': '^P-[0-9]{2}$'}},
            'required': ['procedure_id'],
            'additionalProperties': False,
        },
    },
    {
        'name': 'find_available_crew',
        'description': 'Find an available synthetic crew with an exact qualification code.',
        'parameters': {
            'type': 'object',
            'properties': {'qualification': {'type': 'string', 'enum': ['LV-A', 'HV-B']}},
            'required': ['qualification'],
            'additionalProperties': False,
        },
    },
]
assert all(tool['parameters'].get('additionalProperties') is False for tool in tool_definitions)
print('PASS — tool schemas reject unexpected arguments.')

## 2. Regression cases

`T-02` contains the right function but the wrong qualification. `T-04` proves that no tool call can be the correct behavior. The exact local gate covers `T-04`; the current cloud tool evaluators require at least one tool call, so the cloud run explicitly excludes that one row.

In [ ]:
cases = [
    {
        'case_id': 'T-01',
        'query': 'Inspect synthetic incident SIM-1042.',
        'tool_calls': [{'type': 'tool_call', 'tool_call_id': 't1', 'name': 'lookup_incident', 'arguments': {'incident_id': 'SIM-1042'}}],
        'expected_calls': [{'name': 'lookup_incident', 'arguments': {'incident_id': 'SIM-1042'}}],
        'tool_definitions': tool_definitions,
        'expected_local': 'pass',
    },
    {
        'case_id': 'T-02',
        'query': 'Find an available crew qualified for LV-A work.',
        'tool_calls': [{'type': 'tool_call', 'tool_call_id': 't2', 'name': 'find_available_crew', 'arguments': {'qualification': 'HV-B'}}],
        'expected_calls': [{'name': 'find_available_crew', 'arguments': {'qualification': 'LV-A'}}],
        'tool_definitions': tool_definitions,
        'expected_local': 'fail',
    },
    {
        'case_id': 'T-03',
        'query': 'Look up synthetic procedure P-17.',
        'tool_calls': [{'type': 'tool_call', 'tool_call_id': 't3', 'name': 'lookup_procedure', 'arguments': {'procedure_id': 'P-17'}}],
        'expected_calls': [{'name': 'lookup_procedure', 'arguments': {'procedure_id': 'P-17'}}],
        'tool_definitions': tool_definitions,
        'expected_local': 'pass',
    },
    {
        'case_id': 'T-04',
        'query': 'Explain what an internal-operations assistant is in one sentence.',
        'tool_calls': [],
        'expected_calls': [],
        'tool_definitions': tool_definitions,
        'expected_local': 'pass',
    },
]
assert len({row['case_id'] for row in cases}) == len(cases)
print('Prepared:', [row['case_id'] for row in cases])

## 3. Exact local gate

In [ ]:
def normalize_calls(calls):
    return [
        {'name': call['name'], 'arguments': call.get('arguments', {})}
        for call in calls
    ]

def exact_result(row):
    return 'pass' if normalize_calls(row['tool_calls']) == row['expected_calls'] else 'fail'

baseline_results = {row['case_id']: exact_result(row) for row in cases}
assert baseline_results == {row['case_id']: row['expected_local'] for row in cases}
print('PASS — the local gate found the seeded parameter defect:', baseline_results)

## 4. Semantic Tool Call Accuracy evaluation

Tool Call Accuracy uses the user query, the calls made, and the available tool schemas. It produces an LLM-assisted score and pass/fail label; inspect reasons for disagreements with the exact gate.

In [ ]:
from azure.identity import InteractiveBrowserCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

project_client = AIProjectClient(endpoint=endpoint, credential=InteractiveBrowserCredential())
openai_client = project_client.get_openai_client()
data_source_config = DataSourceConfigCustom(
    type='custom',
    item_schema={
        'type': 'object',
        'properties': {
            'case_id': {'type': 'string'},
            'query': {'type': 'string'},
            'tool_calls': {'type': 'array'},
            'expected_calls': {'type': 'array'},
            'tool_definitions': {'type': 'array'},
            'expected_local': {'type': 'string'},
        },
        'required': ['case_id', 'query', 'tool_calls', 'tool_definitions'],
    },
)
common_mapping = {
    'query': '{{item.query}}',
    'tool_calls': '{{item.tool_calls}}',
    'tool_definitions': '{{item.tool_definitions}}',
}
testing_criteria = [
    TestingCriterionAzureAIEvaluator(
        type='azure_ai_evaluator',
        name='tool_call_accuracy',
        evaluator_name='builtin.tool_call_accuracy',
        initialization_parameters={'deployment_name': model_deployment},
        data_mapping=common_mapping,
    ),
    TestingCriterionAzureAIEvaluator(
        type='azure_ai_evaluator',
        name='tool_selection',
        evaluator_name='builtin.tool_selection',
        initialization_parameters={'deployment_name': model_deployment},
        data_mapping=common_mapping,
    ),
]
print('Configured semantic evaluators.')


In [ ]:
cloud_cases = [row for row in cases if row['tool_calls']]
assert {row['case_id'] for row in cloud_cases} == {'T-01', 'T-02', 'T-03'}
print('Cloud evaluator rows:', [row['case_id'] for row in cloud_cases])

eval_name = f'd2-tool-regression-{resource_namespace}'
run_name = f'd2-tool-regression-baseline-{resource_namespace}'
eval_object = openai_client.evals.create(
    name=eval_name,
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=run_name,
    metadata={'namespace': resource_namespace, 'team_id': team_id, 'suite': 'tool-regression-v1'},
    data_source={
        'type': 'jsonl',
        'source': {'type': 'file_content', 'content': [{'item': row} for row in cloud_cases]},
    },
)
print({'evaluation_id': eval_object.id, 'run_id': eval_run.id})

In [ ]:
deadline = time.monotonic() + 20 * 60
while eval_run.status not in ('completed', 'failed', 'canceled'):
    if time.monotonic() > deadline:
        raise TimeoutError('Evaluation exceeded 20 minutes')
    time.sleep(5)
    eval_run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    print('status:', eval_run.status)
output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
if eval_run.status != 'completed':
    run_error = getattr(eval_run, 'error', None)
    if hasattr(run_error, 'model_dump'):
        run_error = run_error.model_dump(mode='json')
    raise RuntimeError(f'Evaluation infrastructure failure: {{"status": {eval_run.status!r}, "eval_id": {eval_object.id!r}, "run_id": {eval_run.id!r}, "server_error": {run_error!r}, "output_items": {len(output_items)}, "report_url": {getattr(eval_run, "report_url", None)!r}}}')
output_deadline = time.monotonic() + 2 * 60
while len(output_items) < len(cloud_cases):
    if time.monotonic() > output_deadline:
        raise TimeoutError(f'Evaluation completed but exposed only {len(output_items)}/{len(cloud_cases)} output items')
    time.sleep(2)
    output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
assert len(output_items) == len(cloud_cases)
failed_results = [
    result
    for item in output_items
    for result in item.model_dump(mode='json')['results']
    if result.get('error') or result.get('status') in ('failed', 'error', 'canceled')
]
assert not failed_results, failed_results
for item in output_items:
    print(item.model_dump(mode='json') if hasattr(item, 'model_dump') else item)
print({'report_url': getattr(eval_run, 'report_url', None), 'output_items': len(output_items)})
print('PASS — semantic tool-call results are available for review.')

## Participant challenge

Create `fixed_cases` without mutating the baseline. Correct the `T-02` qualification, make every exact assertion pass, and submit a second run named `d2-tool-regression-fixed-<namespace>`.

Then add one negative case where the agent calls a tool for a question that requires no tool. Explain whether Tool Selection or Tool Call Accuracy gives the clearer diagnostic.

In [ ]:
# TODO: build and validate the repaired suite.
# fixed_cases = ...
# assert all(exact_result(row) == 'pass' for row in fixed_cases)
# fixed_run = openai_client.evals.runs.create(...)


## Optional extension

Add Tool Input Accuracy for strict schema diagnostics and Task Navigation Efficiency for known multi-step paths. Use deterministic mocks in CI and reserve live operational integrations for a controlled test environment.

## Cleanup (opt-in and namespace-safe)

In [ ]:
if os.getenv('WORKSHOP_ALLOW_CLEANUP', 'false').lower() == 'true':
    if not eval_name.endswith(f'-{resource_namespace}'):
        raise RuntimeError(f'Refusing to delete non-owned evaluation: {eval_name}')
    openai_client.evals.delete(eval_id=eval_object.id)
    print('Deleted this namespaced evaluation.')
else:
    print('Cleanup disabled; keep baseline and fixed reports for comparison.')